# Qwen anchor long retention compare (complex math, Colab)

Этот notebook запускает long-generation compare только для **сложного математического prompt**:
- base `Qwen/Qwen2.5-1.5B`
- anchor-biased `Qwen/Qwen2.5-1.5B`

Используется новая схема **entropy-pressure gated anchor bias** с `pressure_rescue_floor`,
чтобы bias не обнулялся на low-entropy confident-error шагах.


> Рекомендуемый runtime в Colab: **GPU**.
>
> Это long-generation run, поэтому на CPU будет очень медленно.


In [ ]:
!nvidia-smi || true
!python --version
!pip install -q torch transformers einops pytest


In [ ]:
%cd /content
import os
if not os.path.exists('/content/ABPT'):
    !git clone https://github.com/kharkilirov1/Anchor-engine.git ABPT
%cd /content/ABPT
!git fetch --all
!git checkout main
!git pull --ff-only origin main


In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-1.5B'
DEVICE = 'cuda'
MAX_NEW_TOKENS = 500
MAX_LENGTH = 1024
SEED = 7

CONFLICT_THRESHOLD = 0.55
BIAS_SCALE = 1.50
REPETITION_PENALTY = 1.15
FREQUENCY_PENALTY = 0.05
NO_REPEAT_NGRAM_SIZE = 3
MAX_BIAS_GATE_SUM = 1.50

ENTROPY_TOP_K = 32
ENTROPY_THRESHOLD = 0.35
ENTROPY_SLOPE = 0.08
PRESSURE_THRESHOLD = 0.60
PRESSURE_SLOPE = 0.08
PRESSURE_RESCUE_FLOOR = 0.20

PROMPT = (
    'Prove that sqrt(2) + sqrt(3) is irrational using proof by contradiction only. '
    'Start by assuming the sum is rational, derive the consequences step by step, '
    'and end with an explicit contradiction. '
    'Do not use field extensions, minimal polynomials, algebraic degree arguments, or shortcuts.'
)

POSITIVE_KEYWORDS = (
    'proof by contradiction,assume the sum is rational,assume rational,contradiction,'
    'square both sides,isolate the radical,irrational'
)
NEGATIVE_KEYWORDS = (
    'field extension,minimal polynomial,algebraic degree,galois,shortcut,well-known result'
)

OUTPUT_JSON = 'archive/colab_qwen_complex_math_long_compare.json'
OUTPUT_MD = 'docs/research/colab_qwen_complex_math_long_compare.md'


In [ ]:
import subprocess
import sys
from pathlib import Path

cmd = [
    sys.executable,
    'scripts/run_qwen_long_retention_compare.py',
    '--model', MODEL_NAME,
    '--device', DEVICE,
    '--prompt', PROMPT,
    '--max_new_tokens', str(MAX_NEW_TOKENS),
    '--max_length', str(MAX_LENGTH),
    '--seed', str(SEED),
    '--conflict_threshold', str(CONFLICT_THRESHOLD),
    '--bias_scale', str(BIAS_SCALE),
    '--repetition_penalty', str(REPETITION_PENALTY),
    '--frequency_penalty', str(FREQUENCY_PENALTY),
    '--no_repeat_ngram_size', str(NO_REPEAT_NGRAM_SIZE),
    '--max_bias_gate_sum', str(MAX_BIAS_GATE_SUM),
    '--entropy_top_k', str(ENTROPY_TOP_K),
    '--entropy_threshold', str(ENTROPY_THRESHOLD),
    '--entropy_slope', str(ENTROPY_SLOPE),
    '--pressure_threshold', str(PRESSURE_THRESHOLD),
    '--pressure_slope', str(PRESSURE_SLOPE),
    '--pressure_rescue_floor', str(PRESSURE_RESCUE_FLOOR),
    '--positive_keywords', POSITIVE_KEYWORDS,
    '--negative_keywords', NEGATIVE_KEYWORDS,
    '--output_json', OUTPUT_JSON,
    '--output_md', OUTPUT_MD,
]
print('RUNNING:', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'compare script failed with code {result.returncode}')
assert Path(OUTPUT_JSON).exists(), f'missing output json: {OUTPUT_JSON}'
assert Path(OUTPUT_MD).exists(), f'missing output md: {OUTPUT_MD}'


In [ ]:
import json
from pathlib import Path

payload = json.loads(Path(OUTPUT_JSON).read_text(encoding='utf-8'))
print('base lexical score:', payload['base_analysis']['lexical_score'])
print('anchor lexical score:', payload['anchor_analysis']['lexical_score'])
print('base positive hits:', payload['base_analysis']['positive_hits'])
print('anchor positive hits:', payload['anchor_analysis']['positive_hits'])
print('base negative hits:', payload['base_analysis']['negative_hits'])
print('anchor negative hits:', payload['anchor_analysis']['negative_hits'])
print('anchor bias active steps:', sum(1 for step in payload['anchor']['steps'] if step.get('bias_applied')))
print('mean alpha_t:', sum(step.get('effective_bias_scale', 0.0) for step in payload['anchor']['steps']) / max(len(payload['anchor']['steps']), 1))
print('mean entropy:', sum(step.get('normalized_entropy', 0.0) for step in payload['anchor']['steps']) / max(len(payload['anchor']['steps']), 1))
print('mean entropy gate:', sum(step.get('entropy_gate', 0.0) for step in payload['anchor']['steps']) / max(len(payload['anchor']['steps']), 1))
print('mean pressure gate:', sum(step.get('pressure_gate', 0.0) for step in payload['anchor']['steps']) / max(len(payload['anchor']['steps']), 1))


In [ ]:
print('=== BASE CONTINUATION ===')
print(payload['base']['continuation_text'][:6000])
print()
print('=== ANCHOR CONTINUATION ===')
print(payload['anchor']['continuation_text'][:6000])


In [ ]:
for idx, step in enumerate(payload['anchor']['steps'][:40]):
    print(idx, {
        'token_text': step.get('token_text'),
        'alpha_t': step.get('effective_bias_scale'),
        'entropy': step.get('normalized_entropy'),
        'entropy_gate': step.get('entropy_gate'),
        'pressure_gate': step.get('pressure_gate'),
        'bias_applied': step.get('bias_applied'),
        'num_active': step.get('num_active'),
        'mean_pressure': step.get('mean_contradiction_pressure'),
    })


Этот notebook намеренно заточен только под **один сложный математический кейс**.
Если потом понадобится ещё один сложный math prompt, лучше добавить вторую notebook-версию, а не смешивать несколько задач здесь.
